In [1]:
import joblib
import os
import numpy as np
from tensorflow.keras.models import load_model
import time

print("--- Step 1: Initializing the SENTRi-X Hybrid Engine ---")

processed_dir = "../data/processed/"
models_dir = "../models/"

# 1. Load the unseen testing data
print("Loading unseen testing matrices...")
X_test = joblib.load(os.path.join(processed_dir, "ton_iot_X_test.pkl"))
y_test = joblib.load(os.path.join(processed_dir, "ton_iot_y_test.pkl"))

# 2. Reshape the data for the CNN specifically
X_test_cnn = np.array(X_test).reshape(X_test.shape[0], X_test.shape[1], 1)

# 3. Load both trained models
print("Waking up the Random Forest baseline...")
rf_model = joblib.load(os.path.join(models_dir, "rf_model_ton_iot.joblib"))

print("Waking up the Convolutional Neural Network...")
cnn_model = load_model(os.path.join(models_dir, "cnn_model_ton_iot.h5"))

print("\nAll systems online. Ready for Ensemble Fusion.")

--- Step 1: Initializing the SENTRi-X Hybrid Engine ---
Loading unseen testing matrices...
Waking up the Random Forest baseline...
Waking up the Convolutional Neural Network...



All systems online. Ready for Ensemble Fusion.


In [2]:
print("--- Step 2: Executing Ensemble Fusion (Soft Voting) ---")

start_inference = time.time()

# 1. Get exact probabilities from the Random Forest
# predict_proba returns [Probability of 0, Probability of 1]. We only want the 1 (Attack).
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# 2. Get exact probabilities from the CNN
# The CNN already outputs a single probability for the 1 class. We flatten it to a standard 1D array.
cnn_probabilities = cnn_model.predict(X_test_cnn, verbose=0).flatten()

# 3. The Fusion: Average the mathematical confidences
hybrid_probabilities = (rf_probabilities + cnn_probabilities) / 2.0

# 4. The Final Decision: If the combined confidence is > 50%, flag it as an attack
hybrid_predictions = (hybrid_probabilities > 0.5).astype("int32")

end_inference = time.time()

print(f"\nHybrid Inference Complete!")
print(f"Time to process {X_test.shape[0]:,} flows through BOTH models: {round(end_inference - start_inference, 4)} seconds")

--- Step 2: Executing Ensemble Fusion (Soft Voting) ---

Hybrid Inference Complete!
Time to process 199,965 flows through BOTH models: 30.3951 seconds


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("--- Step 3: SENTRi-X Hybrid Engine Performance ---")

print(f"Final Ensemble Accuracy Score: {accuracy_score(y_test, hybrid_predictions) * 100:.4f}%\n")

print("Classification Report:")
print(classification_report(y_test, hybrid_predictions, target_names=['Normal (0)', 'Attack (1)'], digits=4))

# Confusion Matrix
cm_hybrid = confusion_matrix(y_test, hybrid_predictions)
print("\nFinal Confusion Matrix Breakdown:")
print(f"True Negatives (Correctly identified Normal): {cm_hybrid[0][0]:,}")
print(f"False Positives (False Alarms):               {cm_hybrid[0][1]:,}")
print(f"False Negatives (Missed Attacks):             {cm_hybrid[1][0]:,}")
print(f"True Positives (Correctly identified Attack): {cm_hybrid[1][1]:,}")